In [5]:
import pymongo
import time
from datetime import datetime

def consistencia():
    print("Inicio prueba consistencia")
    print("="*50)

    nodos = {}
    for puerto in [30001, 30002, 30003]:
        try:
            client = pymongo.MongoClient(f"mongodb://localhost:{puerto}/?directConnection=true", serverSelectionTimeoutMS=3000)
            client.admin.command("ping")
            nodos[puerto] = client
            print(f"{puerto} conectado")
        except:
            print(f"No se pudo conectar a{puerto}")
    
    if len(nodos) < 2:
        print("No se pudo conectar a suficientes nodos para la prueba.")
        return False
    
    primer_nodo = list(nodos.values())[0]
    collection = primer_nodo["Politica"]["TestConsistencia"]
    
    collection.delete_many({})
    doc_id = f"test_{int(time.time())}"
    
    collection.insert_one({
        "_id": doc_id,
        "mensaje": "Prueba de consistencia",
        "timestamp": datetime.now().isoformat()
    })
    
    print(f"Insertado documento: {doc_id}")
    
    time.sleep(3)
    
    print("Verificando en todos los nodos:")
    consistente = True
    
    for puerto, client in nodos.items():
        test_collection = client["Politica"]["TestConsistencia"]
        doc = test_collection.find_one({"_id": doc_id})
        
        if doc:
            print(f"{puerto}: Documento encontrado")
        else:
            print(f"{puerto}: Documento NO encontrado")
            consistente = False
    
    if consistente:
        print("\nConsistente!")
    else:
        print("\nERROR: No consistente!")
    
    collection.delete_many({})
    for client in nodos.values():
        client.close()
    
    return consistente

consistencia()

Inicio prueba consistencia
30001 conectado
30002 conectado
30003 conectado
Insertado documento: test_1763430884
Verificando en todos los nodos:
30001: Documento encontrado
30002: Documento encontrado
30003: Documento encontrado

Consistente!


True

In [6]:
import subprocess

def altaDisponibilidad():
    print("Inicio prueba alta disponibilidad")
    print("="*50)
    
    try:
        client = pymongo.MongoClient("mongodb://localhost:30001,localhost:30002,localhost:30003/?replicaSet=my-replicaset",serverSelectionTimeoutMS=10000)
        client.admin.command("ping")
        print("Conectado al replica set")
    except Exception as e:
        print(f"Error inicial: {e}")
        return False
    
    collection = client["Politica"]["TestDisponibilidad"]
    collection.delete_many({})
    
    doc_id = "test_disponibilidad"
    collection.insert_one({"_id": doc_id, "estado": "inicial"})
    print("Insertado documento inicial")
    
    print("Simulando fallo del nodo mongo1 (primary)...")
    try:
        subprocess.run(["docker", "stop", "mongo1"], capture_output=True, timeout=10)
        print("Nodo mongo1 detenido")
    except Exception as e:
        print(f"Error al detener nodo primary: {e}")
    
    print("Esperando failover automático...")
    time.sleep(10)
    
    try:
        collection.update_one({"_id": doc_id}, {"$set": {"estado": "post_fallo"}})
        doc = collection.find_one({"_id": doc_id})
        
        if doc and doc["estado"] == "post_fallo":
            print("Sistema sigue operativo después del fallo")
            alta_disponibilidad = True
        else:
            print("Sistema no operativo")
            alta_disponibilidad = False
            
    except Exception as e:
        print(f"Error después del fallo: {e}")
        alta_disponibilidad = False
    
    print("\nRestaurando nodo mongo1...")
    try:
        subprocess.run(["docker", "start", "mongo1"], capture_output=True, timeout=15)
        print("Nodo mongo1 restaurado")
        time.sleep(5)
    except Exception as e:
        print(f"Error al restaurar: {e}")
    
    if alta_disponibilidad:
        print("\nAlta disponibilidad!")
    else:
        print("\nERROR: No se mantuvo la alta disponibilidad!")
    
    try:
        collection.delete_many({})
        client.close()
    except:
        pass
    
    return alta_disponibilidad

altaDisponibilidad()

Inicio prueba alta disponibilidad
Conectado al replica set
Insertado documento inicial
Simulando fallo del nodo mongo1 (primary)...
Nodo mongo1 detenido
Esperando failover automático...
Sistema sigue operativo después del fallo

Restaurando nodo mongo1...
Nodo mongo1 restaurado

Alta disponibilidad!


True